# STEP Doghouse 安装面 / 安装孔识别完整管线

本 notebook 用于从一个 CAD STEP 文件端到端识别 doghouse 实例、安装面和安装孔，并输出用于装配的 JSON 与可视化 STEP。

**生产默认已整合**（见 `pipeline_defaults.py`）：无需手动传 `--checkpoint` / `--instance-sim-gallery`，`infer_from_step.py --step <file>` 即启用混合 PMAE FaceGraphGNN + `min-instance-faces=2` + 实例整体相似性过滤。

主要流程：

1. STEP 导入与几何采样
2. 构建 doghouse 图推理输入 `.npz`
3. 自动生成 / 加载 Point-MAE 逐面 embedding（checkpoint 需要时自动启用）
4. FaceGraphGNN 推理 doghouse 实例
5. 实例整体相似性过滤（去除 false-positive 连通块）
6. 规则 + VF2 提取安装面和安装孔
7. 输出装配 JSON 和着色 STEP
8. （可选）与标注 JSON 对比评估 doghouse 检测效果

In [ ]:
# Cell 1 — 参数配置
from pathlib import Path
import sys

ROOT = Path('/home/ps/3D相似性/Point-MAE')
DOGHOUSE_DIR = ROOT / 'doghouse_ai'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(DOGHOUSE_DIR))

from pipeline_defaults import (
    PRODUCTION_GRAPH_CHECKPOINT,
    PRODUCTION_INSTANCE_SIM_GALLERY,
    DEFAULT_PMAE_CKPT,
    DEFAULT_PMAE_FACE_EMB_DIR,
    DEFAULT_NODE_THRESHOLD,
    DEFAULT_EDGE_THRESHOLD,
    DEFAULT_MIN_INSTANCE_FACES,
)

# 输入 STEP，可替换成任意新模型（示例：pillar / 未命名-0981535409815353）
STEP_PATH = DOGHOUSE_DIR / 'step - 副本2' / 'pillar.step'
LABEL_JSON = DOGHOUSE_DIR / 'step - 副本2' / f'{STEP_PATH.stem}_annotation.json'

GRAPH_CHECKPOINT = PRODUCTION_GRAPH_CHECKPOINT
INSTANCE_SIM_GALLERY = PRODUCTION_INSTANCE_SIM_GALLERY
PMAE_CKPT = DEFAULT_PMAE_CKPT
PMAE_FACE_EMB_DIR = DEFAULT_PMAE_FACE_EMB_DIR

OUTPUT_DIR = DOGHOUSE_DIR / 'step_doghouse_workflow' / STEP_PATH.stem
PMAE_INPUT_DIR = OUTPUT_DIR / 'pmae_input'

NODE_THRESHOLD = DEFAULT_NODE_THRESHOLD
EDGE_THRESHOLD = DEFAULT_EDGE_THRESHOLD
MIN_INSTANCE_FACES = DEFAULT_MIN_INSTANCE_FACES
SAMPLE_POINTS_PER_FACE = 64
USE_VF2 = True
EXPORT_COLORED_STEP = True

print(f'ROOT: {ROOT}')
print(f'STEP: {STEP_PATH}')
print(f'LABEL_JSON: {LABEL_JSON} (exists={LABEL_JSON.exists()})')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'GRAPH_CHECKPOINT: {GRAPH_CHECKPOINT}')
print(f'INSTANCE_SIM_GALLERY: {INSTANCE_SIM_GALLERY}')
print(f'PMAE_CKPT: {PMAE_CKPT}')

In [ ]:
# Cell 2 — 环境与文件检查
import torch

required_files = {
    'STEP_PATH': STEP_PATH,
    'GRAPH_CHECKPOINT': GRAPH_CHECKPOINT,
    'INSTANCE_SIM_GALLERY': INSTANCE_SIM_GALLERY,
    'PMAE_CKPT': PMAE_CKPT,
}

for name, path in required_files.items():
    print(f'{name}: {path} -> {path.exists()}')
    if not path.exists():
        raise FileNotFoundError(f'{name} 不存在: {path}')

if not LABEL_JSON.exists():
    print('LABEL_JSON 不存在，跳过 doghouse 检测评估（新模型无标注时正常）')

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Cell 3 — 一键端到端运行（整合生产默认）
# STEP -> PMAE -> GNN -> min-instance-faces -> instance-sim -> 安装面/孔 -> 着色 STEP

assembly_step_arg = ''
if EXPORT_COLORED_STEP:
    assembly_step_arg = f'--assembly-output-step "{OUTPUT_DIR / (STEP_PATH.stem + "_assembly_colored.step")}"'

cmd = f'''
conda run --no-capture-output -n llm python "{DOGHOUSE_DIR / 'infer_from_step.py'}" \
  --step "{STEP_PATH}" \
  --output-dir "{OUTPUT_DIR}" \
  --sample-points-per-face {SAMPLE_POINTS_PER_FACE} \
  --node-threshold {NODE_THRESHOLD} \
  --edge-threshold {EDGE_THRESHOLD} \
  --min-instance-faces {MIN_INSTANCE_FACES} \
  --pmae-input-dir "{PMAE_INPUT_DIR}" \
  --extract-assembly-features \
  {'--use-vf2' if USE_VF2 else ''} \
  {assembly_step_arg}
'''
print(cmd)
!{cmd}

In [ ]:
# Cell 4 — 分阶段计时运行（可选）
# 快速跑通用 Cell 3；分析瓶颈时运行本 cell，步骤与生产管线一致。

import json
import time
import numpy as np
import torch
from doghouse_ai.build_point_dataset import build_dataset
from doghouse_ai.doghouse_assembly_features import export_assembly_colored_step, extract_assembly_features
from doghouse_ai.infer_from_step import EMPTY_LABELS, _ensure_pmae_face_embeddings
from doghouse_ai.infer_graph import infer_graph_npz
from doghouse_ai.pipeline_defaults import apply_graph_postprocess
from doghouse_ai.step_geometry import build_geometry_from_step

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIMING_REPORT = OUTPUT_DIR / f'{STEP_PATH.stem}_timing_report.json'
GEOMETRY_JSON = OUTPUT_DIR / f'{STEP_PATH.stem}_doghouse_infer.json'
POINTS_NPZ = OUTPUT_DIR / f'{STEP_PATH.stem}_doghouse_points.npz'
PRED_JSON = OUTPUT_DIR / f'{STEP_PATH.stem}_doghouse_pred_faces.json'
ASSEMBLY_JSON = OUTPUT_DIR / f'{STEP_PATH.stem}_doghouse_assembly_features.json'
COLORED_STEP = OUTPUT_DIR / f'{STEP_PATH.stem}_assembly_colored.step'

timings = {}

def timed(name, fn):
    t0 = time.perf_counter()
    value = fn()
    timings[name] = time.perf_counter() - t0
    print(f'{name}: {timings[name]:.3f}s')
    return value

geometry = timed('01_STEP导入_几何导出', lambda: build_geometry_from_step(STEP_PATH, sample_points_per_face=SAMPLE_POINTS_PER_FACE))
timed('02_写geometry_json', lambda: GEOMETRY_JSON.write_text(json.dumps(geometry, ensure_ascii=False, indent=2), encoding='utf-8'))

data = timed('03_构建推理npz数据', lambda: build_dataset(geometry, EMPTY_LABELS))
timed('04_保存基础npz', lambda: np.savez_compressed(POINTS_NPZ, **data))

pmae_emb = timed('05_生成或加载PMAE逐面embedding', lambda: _ensure_pmae_face_embeddings(
    data,
    PMAE_FACE_EMB_DIR,
    STEP_PATH.stem,
    PMAE_CKPT,
    PMAE_INPUT_DIR,
    256,
    32,
    False,
))
timed('06_保存带PMAE的npz', lambda: np.savez_compressed(POINTS_NPZ, **data))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
result = timed('07_GNN推理doghouse实例', lambda: infer_graph_npz(
    data,
    GRAPH_CHECKPOINT,
    device=device,
    node_threshold=NODE_THRESHOLD,
    edge_threshold=EDGE_THRESHOLD,
    min_instance_faces=MIN_INSTANCE_FACES,
))
result = timed('08_实例相似性过滤', lambda: apply_graph_postprocess(
    result,
    data,
    instance_sim_gallery=INSTANCE_SIM_GALLERY,
    enable_instance_sim=True,
))
result.update({
    'source_step': str(STEP_PATH),
    'geometry_json': str(GEOMETRY_JSON),
    'npz': str(POINTS_NPZ),
    'checkpoint': str(GRAPH_CHECKPOINT),
    'backbone': 'graph',
    'pmae_face_embeddings': str(pmae_emb),
})
timed('09_写doghouse预测json', lambda: PRED_JSON.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8'))

assembly = timed('10_提取安装面安装孔', lambda: extract_assembly_features(STEP_PATH, result, use_vf2=USE_VF2))
timed('11_写装配json', lambda: ASSEMBLY_JSON.write_text(json.dumps(assembly, ensure_ascii=False, indent=2), encoding='utf-8'))
if EXPORT_COLORED_STEP:
    timed('12_导出着色STEP', lambda: export_assembly_colored_step(STEP_PATH, assembly, COLORED_STEP))

report = {
    'step': str(STEP_PATH),
    'output_dir': str(OUTPUT_DIR),
    'geometry_json': str(GEOMETRY_JSON),
    'points_npz': str(POINTS_NPZ),
    'pmae_face_embeddings': str(pmae_emb),
    'prediction_json': str(PRED_JSON),
    'assembly_json': str(ASSEMBLY_JSON),
    'colored_step': str(COLORED_STEP) if EXPORT_COLORED_STEP else None,
    'pipeline': result.get('pipeline'),
    'num_faces': int(geometry.get('num_faces', len(geometry.get('faces', [])))),
    'num_points': int(data['points'].shape[0]),
    'pred_instances': len(result.get('doghouse_instances', [])),
    'pred_doghouse_faces': sum(1 for row in result.get('face_predictions', []) if row.get('doghouse')),
    'assembly_instances': len(assembly.get('instances', [])),
    'assembly_ok': sum(1 for inst in assembly.get('instances', []) if inst.get('status') == 'ok'),
    'timings': {k: round(v, 6) for k, v in timings.items()},
    'total_s': round(sum(timings.values()), 6),
}
TIMING_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
# Cell 5 — 结果汇总与 doghouse 检测评估
import json
from eval_face_predictions import evaluate_face_predictions

PRED_JSON = OUTPUT_DIR / f'{STEP_PATH.stem}_doghouse_pred_faces.json'
ASSEMBLY_JSON = OUTPUT_DIR / f'{STEP_PATH.stem}_doghouse_assembly_features.json'
TIMING_REPORT = OUTPUT_DIR / f'{STEP_PATH.stem}_timing_report.json'
COLORED_STEP = OUTPUT_DIR / f'{STEP_PATH.stem}_assembly_colored.step'

print('输出文件:')
for path in [PRED_JSON, ASSEMBLY_JSON, TIMING_REPORT, COLORED_STEP]:
    print(f'  {path}: {path.exists()}')

if PRED_JSON.exists():
    pred = json.loads(PRED_JSON.read_text(encoding='utf-8'))
    print('\nDoghouse 推理:')
    print(f"  instances: {len(pred.get('doghouse_instances', []))}")
    print(f"  doghouse faces: {sum(1 for r in pred.get('face_predictions', []) if r.get('doghouse'))}")
    print(f"  backbone: {pred.get('backbone')}")
    print(f"  pipeline: {pred.get('pipeline')}")

if LABEL_JSON.exists() and PRED_JSON.exists():
    report = evaluate_face_predictions(LABEL_JSON, PRED_JSON)
    print('\nDoghouse 检测评估 (vs annotation):')
    print(f"  gt/pred: {report['gt_instances']}/{report['pred_instances']}")
    print(f"  face_iou={report['face_iou']:.4f} recall={report['recall']:.4f} extra={report['extra_count']}")
    for match in report['matches']:
        print(f"    gt#{match['gt']} <- pred#{match['pred']} iou={match['iou']:.4f}")

if ASSEMBLY_JSON.exists():
    assembly = json.loads(ASSEMBLY_JSON.read_text(encoding='utf-8'))
    instances = assembly.get('instances', [])
    print('\n装配特征:')
    print(f"  instances: {len(instances)}")
    print(f"  ok: {sum(1 for inst in instances if inst.get('status') == 'ok')}")
    print(f"  mount faces: {[(inst.get('mount_face') or {}).get('face_idx') for inst in instances]}")
    print(f"  hole groups per instance: {[len(inst.get('hole_groups', [])) for inst in instances]}")

if TIMING_REPORT.exists():
    report = json.loads(TIMING_REPORT.read_text(encoding='utf-8'))
    print('\n耗时:')
    for name, sec in report.get('timings', {}).items():
        print(f'  {name}: {sec:.3f}s')
    print(f"  total: {report.get('total_s'):.3f}s")

## 使用说明

### 替换新 STEP

修改 Cell 1 中的 `STEP_PATH` 即可，例如：

```python
STEP_PATH = DOGHOUSE_DIR / 'step - 副本2' / '未命名-0981535409815353.step'
```

输出目录默认为 `doghouse_ai/step_doghouse_workflow/{stem}/`。

### 生产默认权重（`pipeline_defaults.py`）

| 资源 | 路径 |
|------|------|
| Graph checkpoint | `checkpoints/doghouse_graph_pmae_7_plus_B126302301001.pt` |
| Instance-sim gallery | `checkpoints/doghouse_instance_similarity_gallery_7_plus_B126302301001.npz` |
| PMAE 权重 | `ckpt-last.pth` |
| PMAE embedding 缓存 | `pmae_face_emb/{stem}_pmae_face_emb.npy` |

`infer_from_step.py` 默认：`--backbone graph`、`--min-instance-faces 2`、`--instance-sim-filter` 开启。PMAE 仅在 checkpoint 的 `extra_dim > 0` 时自动启用。

### 主要输出

- `{stem}_doghouse_infer.json`：STEP 几何导出
- `{stem}_doghouse_points.npz`：GNN 推理输入（含 `face_pmae`）
- `{stem}_doghouse_pred_faces.json`：doghouse 实例预测（含 `pipeline` 元数据）
- `{stem}_doghouse_assembly_features.json`：安装面 / 安装孔
- `{stem}_assembly_colored.step`：着色 STEP（绿=安装面，蓝=孔壁，浅红=doghouse）
- `{stem}_timing_report.json`：分阶段耗时（Cell 4）

### 固定回归测试（0981）

对已标注模型 `未命名-0981535409815353`，可运行固定端到端测试：

```bash
python doghouse_ai/run_e2e_test_0981.py          # 完整推理 + 评估
python doghouse_ai/run_e2e_test_0981.py --eval-only  # 仅评估已有输出
python -m unittest doghouse_ai.test_e2e_0981 -v
```

基准：5/5 实例，Face IoU **0.9308**，Recall **1.0**，装配 **ok=5/5**。

### 回退基线

纯面图、关闭 instance-similarity：

```bash
python doghouse_ai/infer_from_step.py --step <file.step> \
  --checkpoint doghouse_ai/checkpoints/doghouse_graph_v1.pt \
  --no-instance-sim-filter
```